# FT-900 原始索引与2012分类断点

只检查官方历史索引、2015-08链接例外及2012分类说明。零市场结果、零拟合、零账户；执行预算与停止条件见 `plan.json`。


In [1]:
import json,hashlib,re,shutil
from pathlib import Path
from bs4 import BeautifulSoup
root=Path.cwd();card=root/'research/census_ft900_index_boundary_20261023';plan=json.loads((card/'plan.json').read_text())
def sha(p):return hashlib.sha256(Path(p).read_bytes()).hexdigest()
source=plan['already_received_index'];index=Path(source['temporary_path'])
assert index.stat().st_size==source['bytes'] and sha(index)==source['sha256']
snapshot=root/'data/snapshots/census_ft900_index_boundary_20261023';snapshot.mkdir(parents=True,exist_ok=True)
index_copy=snapshot/'ft900_index_20261023.html'
if index_copy.exists():assert sha(index_copy)==source['sha256']
else:shutil.copyfile(index,index_copy)
soup=BeautifulSoup(index.read_bytes(),'html.parser')
months='January February March April May June July August September October November December'.split()
number={name:i+1 for i,name in enumerate(months)}
rows=[]
for div in soup.select('div.uscb-list-item-container'):
 p=div.find('p');title=' '.join(p.get_text(' ',strip=True).split()) if p else ''
 m=re.fullmatch(r'([A-Za-z]+) (20(?:1[1-9]|2[0-2]))',title)
 if not m:continue
 raw_month,year=m.group(1),int(m.group(2))
 if raw_month not in number and raw_month!='Ferbruary':continue
 month=number.get(raw_month,2)
 urls=sorted({a['href'] for a in div.select('a[href]') if re.search(r'/ft900_\d{4}\.pdf$',a['href'])})
 assert len(urls)==1,(title,urls)
 expected=f'ft900_{year%100:02d}{month:02d}.pdf'
 rows.append({'year':year,'month':month,'label':title,'pdf_url':urls[0],
              'expected_filename':expected,'linked_filename':urls[0].rsplit('/',1)[-1],
              'link_matches_label':urls[0].endswith('/'+expected)})
assert len(rows)==144 and len({(x['year'],x['month']) for x in rows})==144
mismatches=[x for x in rows if not x['link_matches_label']]
assert len(mismatches)==1 and mismatches[0]['expected_filename']=='ft900_1508.pdf'
print({'monthly_labels':len(rows),'correct_index_links':len(rows)-len(mismatches),'mismatched_links':mismatches})


{'monthly_labels': 144, 'correct_index_links': 143, 'mismatched_links': [{'year': 2015, 'month': 8, 'label': 'August 2015', 'pdf_url': 'https://www.census.gov/foreign-trade/Press-Release/ft900/ft900_1509.pdf', 'expected_filename': 'ft900_1508.pdf', 'linked_filename': 'ft900_1509.pdf', 'link_matches_label': False}]}


In [2]:
from datetime import datetime,timezone
import requests
from pypdf import PdfReader
from io import BytesIO
receipts=[];texts={}
for item in plan['exact_new_GETs']:
 name=item['url'].rsplit('/',1)[-1];dest=snapshot/name
 if dest.exists():raise ValueError('fixed source GET already consumed; do not silently repeat')
 start=datetime.now(timezone.utc).isoformat();response=requests.get(item['url'],timeout=40)
 end=datetime.now(timezone.utc).isoformat()
 assert response.status_code==200 and response.headers.get('Content-Type','').startswith('application/pdf')
 body=response.content;assert body.startswith(b'%PDF-') and len(body)<3_000_000
 dest.write_bytes(body)
 pages=[page.extract_text() or '' for page in PdfReader(BytesIO(body)).pages]
 text='\n'.join(pages);texts[name]=text
 receipts.append({'url':item['url'],'requested_at_utc':start,'received_at_utc':end,'status':response.status_code,
   'date_header':response.headers.get('Date'),'last_modified_header':response.headers.get('Last-Modified'),
   'content_type':response.headers.get('Content-Type'),'bytes':len(body),'sha256':sha(dest),
   'pages':len(pages),'printed_month_found':(bool(re.search(r'August\s+2015',text,re.I)) if name=='ft900_1508.pdf' else bool(re.search(r'February\s+2012',text,re.I))),
   'semiconductors_mentions':len(re.findall('Semiconductors',text,re.I))})
assert all(x['printed_month_found'] and x['semiconductors_mentions'] for x in receipts)
notice=texts['ft900_1202.pdf']
assert 'changes to end-use commodity classification' in notice.lower()
assert 'smart cards' in notice.lower() and 'flash memory cards' in notice.lower()
result={'status':'index_coverage_and_2012_definition_break_verified_source_only','index_sha256':source['sha256'],
 'indexed_month_labels':144,'correct_label_PDF_links':143,'wrong_link_label':mismatches[0],
 'august_2015_official_path_PDF_verified':True,
 'minor_index_label_issues':['May  2018 spacing','Ferbruary 2019 spelling'],
 '2012_classification_notice':'Effective with January 2012 statistics; semiconductor media such as smart cards and flash memory cards reassigned to semiconductors and related devices within capital goods.',
 'raw_PDF_receipts':receipts,'historical_first_public_bytes_qualified':False,'all_144_source_PDFs_qualified':False,
 'cross_2012_absolute_level_comparable':False,'model_input_qualified':False,'market_outcome_reads':0,
 'factor_values':0,'model_fits':0,'native_accounts':0,'SOXX_goal_achieved':False}
(card/'result.json').write_text(json.dumps(result,ensure_ascii=False,indent=2)+'\n')
print({'status':result['status'],'PDFs':[(x['url'].rsplit('/',1)[-1],x['bytes'],x['semiconductors_mentions']) for x in receipts]})


{'status': 'index_coverage_and_2012_definition_break_verified_source_only', 'PDFs': [('ft900_1508.pdf', 1000259, 2), ('ft900_1202.pdf', 395812, 3)]}


In [3]:
import tarfile,tempfile,shlex
from src.data.nas_market_data import _settings,nas_api_call,_ssh
settings=_settings(root)
with tempfile.TemporaryDirectory(prefix='ft900-boundary-') as temp:
 archive=Path(temp)/'source.tar.gz'
 with tarfile.open(archive,'w:gz') as tar:
  for file in sorted(snapshot.iterdir()):
   if file.is_file():tar.add(file,arcname='source/'+file.name)
 digest=sha(archive)
 receipt=nas_api_call(origin=settings['origin'],token=settings['token'],method='PUT',
  path='/api/research/artifacts/'+digest,upload=archive,executor_id='soxl-minute-research')['artifact']
 assert receipt['sha256']==digest and receipt['size_bytes']==archive.stat().st_size
 nas_path=settings['artifact_root'].rstrip('/')+'/'+digest[:2]+'/'+digest[2:4]+'/'+digest
 remote=_ssh(settings['ssh'],'sha256sum '+shlex.quote(nas_path)).split()[0]
 assert remote==digest
result['nas_archive_sha256']=digest;result['nas_archive_bytes']=receipt['size_bytes'];result['nas_remote_hash_verified']=True
(card/'result.json').write_text(json.dumps(result,ensure_ascii=False,indent=2)+'\n')
(card/'nas_binding.json').write_text(json.dumps({'status':'exact_index_and_two_PDFs_NAS_archived_remote_hash_verified',
 'archive_sha256':digest,'archive_bytes':receipt['size_bytes'],'source_file_sha256':{p.name:sha(p) for p in sorted(snapshot.iterdir()) if p.is_file()},
 'nas_artifact':receipt,'remote_hash_verified':True},ensure_ascii=False,indent=2)+'\n')
print({'nas_sha256':digest,'remote_verified':remote==digest})


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


{'nas_sha256': '8568347b32295da05b300665c4771910cbbb9c8cc6861234ba4cf690873bf1a7', 'remote_verified': True}


In [ ]:
from pypdf import PdfReader
checks=[('ft900_1508.pdf','FT-900 (15-08)','OCTOBER 6, 2015','Semiconductors 3,406 3,702 -296'),
        ('ft900_1202.pdf','FT-900 (12-02)','APRIL 12, 2012','Semiconductors 3,621 3,678 -57')]
reports=[]
for name,issue,release,row in checks:
 pages=[p.extract_text() or '' for p in PdfReader(snapshot/name).pages]
 assert issue in pages[0] and release in pages[0]
 hits=[i+1 for i,t in enumerate(pages) if 'Exhibit 7.' in t and row in t]
 assert hits==[13],(name,hits)
 reports.append({'file':name,'printed_issue':issue,'printed_release':release,
                 'Exhibit7_pdf_page':13,'semiconductors_row_as_printed':row})
result=json.loads((card/'result.json').read_text())
result['targeted_readback']=reports
(card/'result.json').write_text(json.dumps(result,ensure_ascii=False,indent=2)+'\n')
summary={'experiment':'census_ft900_index_boundary_20261023',
 'decision':'source_index_exception_and_2012_definition_break_audited_no_factor',
 'objective_achieved':False,'result':'result.json','nas_binding':'nas_binding.json',
 'index_month_labels':144,'correct_index_links':143,'one_official_path_exception_verified':True,
 '2012_semiconductor_media_reclassification':True,'all_144_source_PDFs_qualified':False,
 'historical_first_public_bytes_qualified':False,'model_input_qualified':False,
 'market_outcome_reads':0,'model_fits':0,'native_accounts':0,
 'artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml',
                    'recorder_only_artifacts':[],'raw_source_nas_archive_sha256':result['nas_archive_sha256']}}
(card/'summary.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2)+'\n')
print({'targeted_readback':reports,'decision':summary['decision']})


## 中文结论

官方历史索引对 2011–2022 的 144 个月给出 143 条月份一致的 PDF 链接；2015 年 8 月标签误连 9 月，官方路径上的 8 月原件已按印刷月份、2015-10-06 08:30 EDT 发布声明及 Exhibit 7 半导体行核实。2012 年 2 月原件列明自 2012 年 1 月统计起，智能卡与闪存卡等媒介重分类至半导体相关资本品；跨断点绝对水平不可直接当作同口径。索引和两份 PDF 原字节已存 NAS 并远端校验。零模型、零账户；144 个月原件及其历史首次公开仍未认证，不授权 Stage1 特征或 SOXX 超额结论。详见 `result.json`、`summary.json` 和 `nas_binding.json`。
